In [ ]:
"""
!pip install segmentation-models-pytorch -q
!pip install kaggle -q
"""

import os, zipfile, glob, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("   ⚠  No GPU — Runtime → Change runtime type → T4 GPU")

In [ ]:


import os
from google.colab import files

# Step 1: Upload kaggle.json
print("Upload your kaggle.json file now...")
uploaded = files.upload()

# Step 2: Save it to the right place
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ kaggle.json saved")

# Step 3: Download + unzip dataset
os.makedirs('/content/refuge_data', exist_ok=True)
!kaggle datasets download -d arnavjain1/glaucoma-datasets \
       -p /content/refuge_data --unzip

# Step 4: Confirm files
print("\nFiles found:")
!find /content/refuge_data -type f | head -40

In [ ]:

"""
Run this after the download to confirm the exact folder layout.
The paths you use later MUST match what you see here.
"""

DATA_ROOT = '/content/refuge_data'

# Walk the tree (first 50 files)
for root, dirs, files_list in os.walk(DATA_ROOT):
    dirs.sort()
    level = root.replace(DATA_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:                                   # Only show files 3 levels deep
        for f in sorted(files_list)[:5]:
            print(f"{indent}  {f}")

# ── After running, set these two variables to match your layout ──

# Common layouts seen in this dataset:
#   Layout A (most common):
IMG_DIR  = '/content/refuge_data/REFUGE/Images_Square'   # .jpg fundus images
MASK_DIR = '/content/refuge_data/REFUGE/Masks'           # .bmp or .png masks

#   Layout B (some downloads):
# IMG_DIR  = '/content/refuge_data/Training400/Images'
# MASK_DIR = '/content/refuge_data/Training400/Masks'

# Verify
img_files  = sorted(glob.glob(os.path.join(IMG_DIR,  '*.*')))
mask_files = sorted(glob.glob(os.path.join(MASK_DIR, '*.*')))
print(f"\n✅ Found {len(img_files)} images and {len(mask_files)} masks")
print(f"   Example image : {img_files[0] if img_files else 'NONE FOUND'}")
print(f"   Example mask  : {mask_files[0] if mask_files else 'NONE FOUND'}")

# Show a sample pair
if img_files and mask_files:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sample_img  = Image.open(img_files[0]).convert('RGB')
    sample_mask = Image.open(mask_files[0])
    axes[0].imshow(sample_img);                      axes[0].set_title('Fundus Image');     axes[0].axis('off')
    axes[1].imshow(np.array(sample_mask), cmap='gray'); axes[1].set_title('Ground Truth Mask'); axes[1].axis('off')
    plt.suptitle(f'Sample pair: {os.path.basename(img_files[0])}', fontsize=13)
    plt.tight_layout(); plt.show()

    # Print unique mask values — tells you how classes are encoded
    mask_arr = np.array(sample_mask)
    print(f"\nMask dtype: {mask_arr.dtype}  shape: {mask_arr.shape}")
    print(f"Unique values in mask: {np.unique(mask_arr)}")
    print("""
  Typical REFUGE encoding:
    0   = Background
    128 = Optic Disc
    255 = Optic Cup
  We will remap these to 0 / 1 / 2 for PyTorch.
    """)

In [ ]:
class GlaucomaDataset(Dataset):
    """
    Loads paired (fundus image, segmentation mask) samples.

    Mask remapping  (REFUGE standard):
      pixel value 0   → class 0  (Background)
      pixel value 128 → class 1  (Optic Disc)
      pixel value 255 → class 2  (Optic Cup)
    """

    def __init__(self, img_dir, mask_dir, image_size=256, augment=False):
        self.image_size = image_size
        self.augment    = augment

        # Match images to masks by stem name (e.g. n0001 ↔ n0001)
        img_paths  = sorted(glob.glob(os.path.join(img_dir,  '*.*')))
        mask_paths = sorted(glob.glob(os.path.join(mask_dir, '*.*')))

        # Build stem → path dicts for safe pairing
        def stem(p): return os.path.splitext(os.path.basename(p))[0]
        mask_dict = {stem(p): p for p in mask_paths}

        self.pairs = []
        for ip in img_paths:
            s = stem(ip)
            if s in mask_dict:
                self.pairs.append((ip, mask_dict[s]))

        print(f"Dataset: {len(self.pairs)} matched image-mask pairs")

        self.img_transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        # ── Image ──────────────────────────────────────────
        img = Image.open(img_path).convert('RGB')
        img_tensor = self.img_transform(img)          # [3, H, W] float

        # ── Mask ───────────────────────────────────────────
        mask = Image.open(mask_path).convert('L')     # Grayscale
        mask = mask.resize((self.image_size, self.image_size),
                           Image.NEAREST)             # Nearest to keep class values
        mask_np = np.array(mask, dtype=np.int64)

        # Remap pixel values → class indices
        # Adjust thresholds if your dataset uses different values
        class_mask = np.zeros_like(mask_np)
        class_mask[mask_np > 200]  = 2               # Optic Cup (bright, ~255)
        class_mask[(mask_np > 50) & (mask_np <= 200)] = 1  # Optic Disc (~128)
        # class 0 (background) is already 0

        mask_tensor = torch.from_numpy(class_mask).long()  # [H, W]

        # Optional light augmentation (only for training)
        if self.augment and random.random() > 0.5:
            img_tensor  = transforms.functional.hflip(img_tensor)
            mask_tensor = transforms.functional.hflip(mask_tensor.unsqueeze(0)).squeeze(0)

        return img_tensor, mask_tensor


# Quick test
dataset = GlaucomaDataset(IMG_DIR, MASK_DIR, image_size=256, augment=False)

if len(dataset) > 0:
    sample_img_t, sample_mask_t = dataset[0]
    print(f"Image tensor : {sample_img_t.shape}  dtype={sample_img_t.dtype}")
    print(f"Mask tensor  : {sample_mask_t.shape}  dtype={sample_mask_t.dtype}")
    print(f"Mask classes : {sample_mask_t.unique().tolist()}")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    # Un-normalize for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    disp = (sample_img_t * std + mean).clamp(0, 1).permute(1,2,0).numpy()
    axes[0].imshow(disp);  axes[0].set_title('Loaded Image (normalised→display)'); axes[0].axis('off')
    cmap3 = ListedColormap(['black','orange','yellow'])
    axes[1].imshow(sample_mask_t.numpy(), cmap=cmap3, vmin=0, vmax=2)
    axes[1].set_title('Remapped Mask (0=BG 1=Disc 2=Cup)'); axes[1].axis('off')
    plt.tight_layout(); plt.show()


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.net(x)


class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        dy = x2.size(2) - x1.size(2)
        dx = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [dx//2, dx-dx//2, dy//2, dy-dy//2])
        return self.conv(torch.cat([x2, x1], dim=1))


class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=3):
        super().__init__()
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64,  128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        self.up1   = Up(1024+512, 512)
        self.up2   = Up(512+256,  256)
        self.up3   = Up(256+128,  128)
        self.up4   = Up(128+64,    64)
        self.outc  = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x  = self.up1(x5, x4)
        x  = self.up2(x,  x3)
        x  = self.up3(x,  x2)
        x  = self.up4(x,  x1)
        return self.outc(x)


model = UNet(n_channels=3, n_classes=3).to(device)
total = sum(p.numel() for p in model.parameters())
print(f"✅ U-Net ready — {total/1e6:.1f}M parameters on {device}")

In [ ]:
BATCH_SIZE = 4          # Reduce to 2 if you get CUDA OOM errors
IMAGE_SIZE = 256
VAL_SPLIT  = 0.2        # 20 % validation

full_dataset = GlaucomaDataset(IMG_DIR, MASK_DIR, image_size=IMAGE_SIZE, augment=False)

n_total = len(full_dataset)
n_val   = max(1, int(n_total * VAL_SPLIT))
n_train = n_total - n_val

train_ds, val_ds = torch.utils.data.random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)     # reproducible split
)

# Enable augmentation only on training set
train_ds.dataset.augment = True

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} samples  |  Val: {len(val_ds)} samples")
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

In [ ]:
def compute_iou(pred_mask, true_mask, num_classes=3):
    """
    Mean Intersection-over-Union (mIoU) — the standard segmentation metric.
    IoU per class = TP / (TP + FP + FN)
    """
    ious = []
    pred_mask = pred_mask.view(-1)
    true_mask = true_mask.view(-1)
    for cls in range(num_classes):
        pred_c = (pred_mask == cls)
        true_c = (true_mask == cls)
        inter  = (pred_c & true_c).sum().float()
        union  = (pred_c | true_c).sum().float()
        if union == 0:
            ious.append(torch.tensor(1.0))   # Class absent → perfect
        else:
            ious.append(inter / union)
    return torch.stack(ious).mean().item()


def compute_pixel_accuracy(pred_mask, true_mask):
    """Simple pixel-level accuracy."""
    correct = (pred_mask == true_mask).sum().float()
    total   = true_mask.numel()
    return (correct / total).item()


# ── Hyperparameters ────────────────────────────────────────────────────────
EPOCHS    = 10           # Start here; increase to 30-50 for real training
LR        = 1e-4
N_CLASSES = 3

# ── Loss + Optimiser ──────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()            # Works directly on logits + long masks
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True
)

# ── History for plotting ──────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_acc': []}

print(f"Starting training for {EPOCHS} epochs...")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):

    # ── Training phase ──────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for batch_idx, (imgs, masks) in enumerate(train_loader):
        imgs  = imgs.to(device)                  # [B, 3, H, W]
        masks = masks.to(device)                 # [B, H, W]  long

        optimizer.zero_grad()
        logits = model(imgs)                     # [B, 3, H, W]
        loss   = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validation phase ────────────────────────────────────────
    model.eval()
    val_loss  = 0.0
    val_iou   = 0.0
    val_acc   = 0.0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)

            logits     = model(imgs)
            loss       = criterion(logits, masks)
            val_loss  += loss.item()

            pred_masks = logits.argmax(dim=1)    # [B, H, W]
            val_iou   += compute_iou(pred_masks, masks, N_CLASSES)
            val_acc   += compute_pixel_accuracy(pred_masks, masks)

    val_loss /= len(val_loader)
    val_iou  /= len(val_loader)
    val_acc  /= len(val_loader)

    scheduler.step(val_loss)

    # ── Record ──────────────────────────────────────────────────
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val mIoU: {val_iou:.4f} | "
          f"Val Acc: {val_acc:.4f}")

print("=" * 65)
print("✅ Training complete!")



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training Curves', fontsize=14, fontweight='bold')
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='tomato')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['val_iou'], color='mediumseagreen', marker='o', markersize=4)
axes[1].set_title('Validation mIoU'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].grid(True, alpha=0.3)
axes[1].axhline(0.7, linestyle='--', color='gray', alpha=0.6, label='mIoU = 0.70 target')
axes[1].legend()

axes[2].plot(epochs_range, history['val_acc'], color='mediumpurple', marker='o', markersize=4)
axes[2].set_title('Validation Pixel Accuracy'); axes[2].set_xlabel('Epoch')
axes[2].set_ylim(0, 1); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training curves saved to /content/training_curves.png")

# Summary
best_iou = max(history['val_iou'])
best_acc = max(history['val_acc'])
print(f"\nBest Val mIoU : {best_iou:.4f}")
print(f"Best Val Acc  : {best_acc:.4f}")

In [ ]:
odel.eval()
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
cmap3 = ListedColormap(['#111111', '#FF8C00', '#FFD700'])

def denorm(t):
    """Undo ImageNet normalisation for display."""
    return (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

# Grab one batch from validation
val_iter = iter(val_loader)
imgs_b, masks_b = next(val_iter)

with torch.no_grad():
    logits_b = model(imgs_b.to(device))
    preds_b  = logits_b.argmax(dim=1).cpu()

n_show = min(4, imgs_b.size(0))
fig, axes = plt.subplots(n_show, 3, figsize=(13, 4 * n_show))
if n_show == 1: axes = axes[None, :]

col_titles = ['Input Image', 'Ground Truth Mask', 'Predicted Mask']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

for i in range(n_show):
    axes[i, 0].imshow(denorm(imgs_b[i]))
    axes[i, 1].imshow(masks_b[i].numpy(),  cmap=cmap3, vmin=0, vmax=2)
    axes[i, 2].imshow(preds_b[i].numpy(),  cmap=cmap3, vmin=0, vmax=2)
    iou_i = compute_iou(preds_b[i], masks_b[i])
    acc_i = compute_pixel_accuracy(preds_b[i], masks_b[i])
    axes[i, 2].set_xlabel(f'mIoU={iou_i:.3f}  Acc={acc_i:.3f}', fontsize=10)
    for ax in axes[i]: ax.axis('off')

legend_els = [
    mpatches.Patch(color='#111111', label='Background'),
    mpatches.Patch(color='#FF8C00', label='Optic Disc'),
    mpatches.Patch(color='#FFD700', label='Optic Cup'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=3,
           fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Validation Predictions vs Ground Truth', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/val_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved to /content/val_predictions.png")

In [ ]:
torch.save(model.state_dict(), '/content/unet_glaucoma.pth')
print("✅ Model weights saved to /content/unet_glaucoma.pth")
print("\nTo reload later:")
print("  model = UNet(n_channels=3, n_classes=3)")
print("  model.load_state_dict(torch.load('/content/unet_glaucoma.pth'))")
print("  model.eval()")

print("\n" + "="*60)
print("  FINAL RESULTS SUMMARY")
print("="*60)
print(f"  Epochs trained  : {EPOCHS}")
print(f"  Best Val mIoU   : {best_iou:.4f}   (target: >0.70)")
print(f"  Best Val Acc    : {best_acc:.4f}   (target: >0.85)")
print(f"  Final Val mIoU  : {history['val_iou'][-1]:.4f}")
print(f"  Final Val Acc   : {history['val_acc'][-1]:.4f}")
print("="*60)
print("""
  Interpretation:
  ─────────────
  mIoU < 0.40  →  Model not learning. Check mask remapping (Cell 4).
  mIoU 0.40–0.65 →  Learning but needs more epochs / data.
  mIoU > 0.65  →  Decent segmentation for a quick run.
  mIoU > 0.80  →  Good clinical-grade result (needs 30+ epochs).

  Common fixes if mIoU stays low:
  • Check np.unique(mask_arr) in Cell 3 — adjust remap thresholds
  • Increase EPOCHS to 30
  • Use pretrained encoder (see BONUS CELL below)
  • Reduce LR to 1e-5 if loss oscillates
""")
